<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 4 · Anatomía y calidad de los datos

La semana pasada perfilaste un catálogo musical y dejaste anotado «para después» todo lo que llamaba
la atención: mínimos imposibles, counts desiguales, columnas sospechosas. Hoy es después. La
industria de esta semana es el **streaming**: el catálogo público de Netflix, 7 787 títulos tal cual
salieron al mundo, sin que nadie los ensuciara a propósito. Ahí está todo lo que un archivo real te
va a hacer: nulos por miles —y con estructura—, fechas guardadas como texto con espacios escondidos,
una columna que mezcla «90 min» con «2 Seasons» como si fueran la misma cosa, y un duplicado
invisible al ojo. Vas a poner un porcentaje exacto a cada problema, decidir qué hacer con cada uno,
escribir la bitácora que permite a otra persona auditar tus decisiones y cerrar con una cifra de
control. Esto desbloquea el resto del curso: no se puede analizar lo que no se puede auditar.

> **Hoy haces** · Inventario de calidad del catálogo con el porcentaje exacto de cada problema
> (90 min). Arreglas las fechas con espacios, separas la columna que mezcla dos unidades, decides
> qué hacer con tres clases de nulos y cazas un duplicado escondido detrás de un espacio invisible.
> Produces la bitácora de limpieza y la cifra de control. Cierras encontrando el fallo silencioso de
> un fragmento generado por IA que borra medio catálogo sin dar un solo error.
>
> **Entrega** · Este cuaderno ejecutado, la bitácora de limpieza completa, la cifra de control
> cuadrada y el daño del fragmento con error cuantificado en títulos y porcentajes.
> Nombre de archivo: `lab_04_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datasets públicos de las semanas 1 a 4 viven en sitio/datos/publicos/
CANDIDATOS = [Path("../datos/publicos"), Path("datos/publicos"), Path("sitio/datos/publicos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Inventario: cuantificar en lugar de opinar

«Los datos están sucios» no es un diagnóstico. Un diagnóstico dice **qué problema, cuántas filas y
qué porcentaje**, porque de ahí sale la decisión: un problema que afecta al 0,1 % se trata distinto
que uno que afecta al 30 %. Y antes de tocar nada se guardan las cifras de partida, porque sin ellas
no habrá cifra de control al final.

In [ ]:
crudo = pd.read_csv(DATOS / "netflix_titulos.csv")

# Cifras de partida: se guardan ANTES de cualquier cambio, para poder cuadrar al final.
N_TITULOS_0 = len(crudo)
N_PELICULAS_0 = int((crudo["type"] == "Movie").sum())
N_SERIES_0 = int((crudo["type"] == "TV Show").sum())

print(f"títulos {N_TITULOS_0:,} · películas {N_PELICULAS_0:,} · series {N_SERIES_0:,} "
      f"· columnas {crudo.shape[1]}\n")

radiografia = pd.DataFrame({
    "tipo": crudo.dtypes.astype(str),
    "no nulos": crudo.notna().sum(),
    "nulos": crudo.isna().sum(),
    "% nulos": (crudo.isna().mean() * 100).round(2),
    "valores distintos": crudo.nunique(),
})
print(radiografia.to_string())

In [ ]:
fechas_con_espacio = crudo["date_added"].notna() & (crudo["date_added"] != crudo["date_added"].str.strip())

inventario = pd.DataFrame([
    ("director ausente",                        int(crudo["director"].isna().sum())),
    ("cast ausente",                            int(crudo["cast"].isna().sum())),
    ("country ausente",                         int(crudo["country"].isna().sum())),
    ("date_added ausente",                      int(crudo["date_added"].isna().sum())),
    ("rating ausente",                          int(crudo["rating"].isna().sum())),
    ("date_added con espacio inicial",          int(fechas_con_espacio.sum())),
    ("duration con dos unidades mezcladas",     int(crudo["duration"].notna().sum())),
    ("filas duplicadas exactas",                int(crudo.duplicated().sum())),
], columns=["problema", "filas afectadas"])
inventario["% de las filas"] = (inventario["filas afectadas"] / N_TITULOS_0 * 100).round(2)

con_nulo = crudo.isna().any(axis=1)
print(f"filas con al menos un nulo: {con_nulo.sum():,} ({con_nulo.mean():.1%})\n")

nulos = inventario.iloc[:5].sort_values("filas afectadas", ascending=False, ignore_index=True)
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(nulos["problema"], nulos["% de las filas"], color="#C44E52")
ax.invert_yaxis()
ax.set_title("El nulo más grande (director, 30,7 %) resultará ser el más explicable, no el más grave")
ax.set_xlabel("% de los títulos del catálogo")
for y, valor in enumerate(nulos["% de las filas"]):
    ax.text(valor + 0.4, y, f"{valor:.2f} %", va="center", fontsize=9)
plt.tight_layout()
plt.show()

inventario

📌 Tres cosas para leer con lápiz. Primera: **el 38,3 % de los títulos tiene al menos un nulo**;
quien «borre las filas incompletas» borra cuatro de cada diez. Segunda: la fila de duplicados
exactos da **cero**, y ese cero también es información —se comprobó y no había—, aunque todavía no
es la última palabra. Tercera: `duration` aparece con el 100 % de las filas afectadas, y no es una
errata: la columna entera tiene un problema que ningún conteo de nulos puede ver. Vamos por partes.

## 2. Tipos de dato: `dtypes` antes que nada

Once de las doce columnas son `object` —texto— y solo `release_year` llegó como número. Dos de esas
columnas de texto mienten: `date_added` debería ser fecha y `duration` debería ser número. Ninguna
de las dos lanza error si las dejas como están; simplemente todo lo que calcules con ellas será
falso o imposible.

In [ ]:
print(crudo.dtypes.to_string())
print()
print("show_id como texto es CORRECTO: 's1486' es una etiqueta, no una cantidad.")
print("date_added como texto es un problema : no se puede ordenar ni restar.")
print("duration como texto es DOS problemas :", crudo["duration"].iloc[1], "y",
      crudo["duration"].iloc[0], "viven en la misma columna.")

## 3. Las fechas: texto con espacios escondidos

`date_added` viene como `"August 14, 2020"`. La conversión parece de una línea… y la línea
explota.

In [ ]:
try:
    pd.to_datetime(crudo["date_added"], format="%B %d, %Y")
except ValueError as e:
    print("ERROR :", str(e)[:80], "…")

El mensaje lo dice con comillas: `" August 4, 2017"` no coincide con el formato **porque empieza
con un espacio**. La salida fácil es `errors="coerce"`: que lo que no se pueda convertir quede en
`NaT` y el código siga. Mira lo que cuesta esa comodidad.

In [ ]:
ingenuo = pd.to_datetime(crudo["date_added"], format="%B %d, %Y", errors="coerce")

print(f"nulos reales en date_added            : {crudo['date_added'].isna().sum()}")
print(f"NaT después de coerce sin limpiar     : {ingenuo.isna().sum()}")
print(f"fechas REALES borradas por el espacio : {ingenuo.isna().sum() - crudo['date_added'].isna().sum()}")
print()
print("así se ve una de las culpables, con repr() para delatar el espacio:")
culpable = crudo.loc[fechas_con_espacio, "date_added"].iloc[0]
print(f"  {culpable!r}")

⚠️ **`errors="coerce"` borra 88 fechas reales sin avisar.** El dato estaba, completo y correcto;
solo tenía un espacio delante. La solución correcta limpia primero y convierte después, y termina
con una verificación: los únicos `NaT` admisibles son los 10 nulos que ya venían en el archivo.

In [ ]:
catalogo = crudo.copy()
catalogo["fecha_agregado"] = pd.to_datetime(catalogo["date_added"].str.strip(), format="%B %d, %Y")

print(f"tipo {catalogo['fecha_agregado'].dtype} · NaT {catalogo['fecha_agregado'].isna().sum()} "
      f"(los 10 nulos originales, ni uno más)")
print(f"rango: {catalogo['fecha_agregado'].min():%d-%m-%Y} a {catalogo['fecha_agregado'].max():%d-%m-%Y}")
print()
print("títulos añadidos por año (últimos cinco):")
print(catalogo["fecha_agregado"].dt.year.value_counts().sort_index().tail(5).to_string())

Rango coherente —de enero de 2008 a enero de 2021— y un negocio que se ve en tres líneas: el
catálogo explota a partir de 2016 y toca techo en 2019 con 2 153 títulos añadidos. La verificación
del rango y del conteo de `NaT` es obligatoria: es lo único que distingue «convertí las fechas» de
«convertí las fechas bien».

## 4. `duration`: dos unidades en la misma columna

El problema de tipos perfecto. La columna dice `"90 min"` para películas y `"2 Seasons"` para
series: texto siempre, y aunque lograras extraer el número, **90 minutos y 2 temporadas no se pueden
promediar**. Primero comprueba si la mezcla tiene estructura.

In [ ]:
formato = np.where(crudo["duration"].str.contains("min"), "minutos", "temporadas")
pd.crosstab(crudo["type"], formato, margins=True)

La tabla es perfecta: **5 377 duraciones en minutos y las 5 377 son películas; 2 410 en temporadas
y las 2 410 son series.** La mezcla no es ruido, es un diseño: alguien metió dos variables distintas
en una columna porque «las dos son la duración». **Decisión: separarla en dos columnas, una por
unidad, y que cada una sea numérica.** Una columna que necesita un `if` para saber qué significa no
es una columna: son dos disfrazadas.

In [ ]:
numero = catalogo["duration"].str.extract(r"(\d+)")[0].astype(int)
es_pelicula = catalogo["type"] == "Movie"

catalogo["duracion_min"] = numero.where(es_pelicula)
catalogo["temporadas"] = numero.where(~es_pelicula)

pelis = catalogo.loc[es_pelicula, "duracion_min"]
series = catalogo.loc[~es_pelicula, "temporadas"]
print(f"películas : mediana {pelis.median():.0f} min · rango {pelis.min():.0f} a {pelis.max():.0f} min")
print(f"series    : mediana {series.median():.0f} temporada · máximo {series.max():.0f}")
print(f"series de una sola temporada : {(series == 1).sum():,} de {len(series):,} ({(series == 1).mean():.0%})")
print(f"nulos en las columnas nuevas : {int(catalogo['duracion_min'].isna().sum()):,} en duracion_min (las series) "
      f"y {int(catalogo['temporadas'].isna().sum()):,} en temporadas (las películas): uno por fila, por diseño")

Ahora sí hay números: la película mediana dura 98 minutos (el rango va de 3 a 312), la serie
mediana tiene una temporada y **el 67 % de las series solo tiene una**. Cada columna nueva es nula
exactamente donde debe serlo: `duracion_min` en las series y `temporadas` en las películas. Ese nulo
no es un problema de calidad: es el diseño correcto diciéndote «esta medida no aplica aquí».

## 5. Nulos con estructura: ausente no siempre significa lo mismo

La semana pasada los nulos eran cinco celdas; aquí son miles. Antes de decidir qué hacer con ellos,
pregunta **dónde viven**. El caso de `director`:

In [ ]:
print("director ausente, por tipo de título:")
print((crudo.groupby("type")["director"].apply(lambda s: s.isna().mean() * 100).round(1)).to_string())
print()
print("cast ausente   :", int(crudo["cast"].isna().sum()), f"({crudo['cast'].isna().mean():.2%})")
print("country ausente:", int(crudo["country"].isna().sum()), f"({crudo['country'].isna().mean():.2%})")

El nulo de `director` tiene estructura: falta en el **92,4 % de las series** y solo en el 3,0 % de
las películas. No es un fallo de captura: una serie tiene un director por episodio, no uno solo, y
quien diseñó la tabla decidió dejar el campo vacío. Ese nulo significa *«no aplica»*, y **un nulo
que significa «no aplica» no se imputa ni se borra: se documenta**.

Tres decisiones distintas para tres nulos distintos:

- **`director` (30,7 %): se conserva nulo.** Es estructural; inventarle un director a 2 389 títulos
  sería fabricar datos.
- **`country` (6,5 %): centinela `Desconocido`.** El país existe, no se registró. El centinela evita
  que un `groupby` por país esconda 507 títulos en silencio.
- **`cast` (9,2 %): se conserva nulo y pasa a limitación conocida.** No hay fuente para imputarlo y
  ningún análisis del curso lo necesita.

In [ ]:
catalogo["country"] = catalogo["country"].fillna("Desconocido")

print(f"nulos restantes en country : {catalogo['country'].isna().sum()}")
print()
print("los cinco países con más títulos, ya con el centinela a la vista:")
print(catalogo["country"].value_counts().head(5).to_string())

`Desconocido` queda tercero, entre India y Reino Unido: si se hubiera quedado como nulo, todo
ranking de países lo habría omitido sin decirlo, y el 6,5 % del catálogo habría desaparecido de los
reportes.

## 6. Duplicados: el que no se ve

El inventario dijo «cero duplicados exactos». Pero un duplicado real casi nunca es exacto: basta un
espacio, una mayúscula o una tilde para que dos filas idénticas parezcan distintas. La comprobación
seria normaliza primero.

In [ ]:
print(f"duplicados exactos                      : {crudo.duplicated().sum()}")
print(f"títulos repetidos tal cual              : {crudo['title'].duplicated().sum()}")

titulo_norm = catalogo["title"].str.strip().str.lower()
print(f"títulos repetidos tras normalizar       : {titulo_norm.duplicated().sum()}")
print()
par = catalogo[titulo_norm.isin(titulo_norm[titulo_norm.duplicated()])]
print(par[["show_id", "type", "title", "country", "release_year", "duration"]].to_string())
print()
print("los dos títulos, con repr() para ver lo invisible:")
for t in par["title"]:
    print(f"  {t!r}")

Ahí está: la película *Consequences* existe dos veces, `s1486` y `s1487`, idénticas en tipo, país,
año, duración, reparto y descripción. La segunda lleva al final del título un **espacio no separable**
(`\xa0`), invisible en pantalla y suficiente para burlar cualquier `duplicated()` ingenuo.
**Decisión: se elimina la copia**, y no «porque los duplicados se borran», sino porque las diez
columnas de negocio coinciden y el identificador distinto no aporta información. Si coincidiera solo
el título, la decisión sería la contraria: dos películas pueden llamarse igual.

In [ ]:
antes = len(catalogo)
catalogo = catalogo[~titulo_norm.duplicated(keep="first")].copy()

print(f"títulos antes  : {antes:,}")
print(f"títulos después: {len(catalogo):,}  (se eliminó 1 copia, el 0,01 % del catálogo)")

## 7. La bitácora de limpieza

Una limpieza sin bitácora no es reproducible: es una opinión. Una fila por problema, siempre las
mismas columnas, incluida la de justificación, que es la que casi nadie escribe.

In [ ]:
bitacora = pd.DataFrame([
    dict(problema="date_added con espacio inicial", filas=88,
         decision="str.strip() antes de convertir con formato explícito",
         justificacion="errors='coerce' sin limpiar borraba 88 fechas reales en silencio"),
    dict(problema="date_added ausente", filas=10,
         decision="Se conserva NaT y se documenta",
         justificacion="No hay fuente para imputar cuándo se añadió el título"),
    dict(problema="duration con dos unidades", filas=7787,
         decision="Separar en duracion_min (películas) y temporadas (series)",
         justificacion="El promedio de '90 min' y '2 Seasons' no existe; una columna, una unidad"),
    dict(problema="director ausente", filas=2389,
         decision="Se conserva nulo: es estructural, no un fallo",
         justificacion="Falta en el 92,4 % de las series: significa 'no aplica', no 'se perdió'"),
    dict(problema="country ausente", filas=507,
         decision="Centinela 'Desconocido'",
         justificacion="Evita que un groupby por país esconda el 6,5 % del catálogo"),
    dict(problema="cast ausente", filas=718,
         decision="Se conserva nulo; limitación conocida",
         justificacion="Sin fuente externa no hay imputación defendible"),
    dict(problema="rating ausente", filas=7,
         decision="Se conserva nulo y se documenta",
         justificacion="0,09 %: ni sesga un análisis ni justifica inventar clasificaciones"),
    dict(problema="título duplicado por espacio invisible", filas=1,
         decision="Eliminar la copia s1487",
         justificacion="Idéntica en las diez columnas de negocio; solo difiere un \\xa0 en el título"),
])
bitacora["% del catálogo"] = (bitacora["filas"] / N_TITULOS_0 * 100).round(2)
bitacora

## 8. La cifra de control

La pregunta que cierra cualquier limpieza es: **¿perdí información?** No se contesta con una
sensación. Se contesta con invariantes: cosas que tenían que quedar exactamente igual, y cambios que
tienen que estar explicados hasta la última fila.

In [ ]:
control = pd.DataFrame([
    ("títulos",              N_TITULOS_0,    len(catalogo)),
    ("películas",            N_PELICULAS_0,  int((catalogo["type"] == "Movie").sum())),
    ("series",               N_SERIES_0,     int((catalogo["type"] == "TV Show").sum())),
    ("sin fecha de adición", int(crudo["date_added"].isna().sum()),
                             int(catalogo["fecha_agregado"].isna().sum())),
    ("sin país",             int(crudo["country"].isna().sum()),
                             int((catalogo["country"] == "Desconocido").sum())),
], columns=["invariante", "antes", "después"])
control["cambio"] = control["después"] - control["antes"]
print(control.to_string(index=False))
print()

peliculas_ok = catalogo["duracion_min"].notna().sum() == (catalogo["type"] == "Movie").sum()
series_ok = catalogo["temporadas"].notna().sum() == (catalogo["type"] == "TV Show").sum()
print(f"¿cada película tiene sus minutos?    : {peliculas_ok}")
print(f"¿cada serie tiene sus temporadas?    : {series_ok}")
print(f"única fila eliminada                 : la copia de 'Consequences' (show_id s1487)")

In [ ]:
# ✅ Comprobación 1 · la limpieza no perdió nada que no esté explicado
assert len(catalogo) == 7_786, \
    f"El catálogo limpio debería tener 7 786 títulos y tiene {len(catalogo):,}: la única baja admisible es s1487"
assert int(catalogo["fecha_agregado"].isna().sum()) == 10, \
    "Deberían quedar exactamente 10 fechas NaT (las originales). Si ves 98, convertiste sin str.strip()"
assert catalogo["duracion_min"].notna().sum() == (catalogo["type"] == "Movie").sum() and \
       catalogo["temporadas"].notna().sum() == (catalogo["type"] == "TV Show").sum(), \
    "duracion_min debe cubrir exactamente las películas y temporadas exactamente las series: revisa el where()"

print(f"Comprobación 1 superada ✓  {len(catalogo):,} títulos auditables, cada cambio explicado")

Ese es el criterio: no «quedó limpio», sino **«todo lo que cambió está explicado y todo lo que no
tenía que cambiar sigue igual»**. Se fue una fila —identificada por su `show_id`—, ningún título
perdió su fecha, su duración quedó en la columna y la unidad correctas, y los nulos que quedan son
decisiones escritas, no descuidos.

### 🌶️ Ejercicio 1 — Guiado

Repite el inventario sobre las dos columnas que hoy quedaron fuera: `rating` y `listed_in`. Para
`rating`: cuántos valores distintos hay y si alguno parece un error de captura más que una
clasificación. Para `listed_in`: cuántos géneros tiene un título típico y cuál es el género más
frecuente del catálogo. Añade a la bitácora una fila por cada problema nuevo, con decisión y
justificación.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: catalogo["rating"].value_counts(dropna=False) — mira los valores con poquísimas filas
# Pista 2: catalogo["listed_in"].str.split(", ").str.len() te da géneros por título
# Pista 3: para el género más frecuente: .str.split(", ").explode().value_counts()

### 🔥 Desafío — el fragmento generado por IA

Este es el código que devolvió un asistente al prompt *«escribe una función que limpie mi catálogo
de streaming en pandas»*. Corre sin errores, es legible y tiene buen aspecto. Contiene **un fallo
grave y uno sutil**. Ejecútalo, encuéntralos y cuantifica el daño: cuántos títulos desaparecen, de
qué tipo son y qué análisis quedan sesgados para siempre.

In [ ]:
def limpiar_catalogo(df):
    """Limpieza básica de un catálogo de streaming."""
    df = df.copy()
    df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce")
    df = df.dropna()           # fuera filas incompletas
    df = df.drop_duplicates()  # fuera duplicados
    return df


catalogo_ia = limpiar_catalogo(pd.read_csv(DATOS / "netflix_titulos.csv"))
print(f"{len(catalogo_ia):,} títulos, sin un solo nulo y sin duplicados. Impecable.")
catalogo_ia.head(3)

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: compara len(catalogo_ia) con len(crudo). ¿Cuántos títulos desaparecieron y qué % es?
# Pista 2: catalogo_ia["type"].value_counts() contra crudo["type"].value_counts().
#          ¿Qué le pasó a las series, y por qué justo a ellas? (sección 5)
# Pista 3: el fallo sutil está en la línea de las fechas: ¿cuántos NaT fabricó el coerce
#          sin strip, y qué hizo el dropna con ellos?
# Entrega: tres cifras — títulos perdidos, % de series perdidas, fechas reales borradas.

### 🎯 Reto en clase (15 min)

En equipos. Cada equipo escribe **una** línea de limpieza plausible pero equivocada sobre el
catálogo —del estilo de las del fragmento anterior— y se la pasa al equipo de al lado. El equipo
receptor tiene diez minutos para encontrar el daño y ponerle una cifra. Gana el equipo cuya línea
tarde más en ser cuantificada, no el que la esconda mejor: si nadie logra medir el daño, la línea no
valía nada como trampa.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: la trampa buena no lanza excepciones. Candidatas: fillna("") sobre country antes de
# contar países, astype(int) sobre temporadas con nulos, un filtro por fecha aplicado antes
# de convertir date_added, o un drop_duplicates(subset=["title"]) sin normalizar ni mirar.

## La trampa de hoy

⚠️ **`dropna()` sin `subset`: la línea más peligrosa del pandas generado por IA.** No falla nunca,
produce una tabla perfecta y decide por ti que toda fila incompleta sobra. En este catálogo el nulo
de `director` es estructural —las series casi nunca lo tienen—, así que «borrar filas incompletas»
significa, en la práctica, **borrar las series**. Los números del fragmento, lado a lado.

In [ ]:
resumen = pd.DataFrame({
    "catálogo real": [N_TITULOS_0, N_PELICULAS_0, N_SERIES_0, N_SERIES_0 / N_TITULOS_0 * 100],
    "tras el fragmento con dropna()": [len(catalogo_ia),
                                       int((catalogo_ia["type"] == "Movie").sum()),
                                       int((catalogo_ia["type"] == "TV Show").sum()),
                                       (catalogo_ia["type"] == "TV Show").mean() * 100],
}, index=["títulos", "películas", "series", "% de series en el catálogo"])
resumen["daño"] = resumen.iloc[:, 1] - resumen.iloc[:, 0]
print(resumen.round(1).to_string())
print()

series_perdidas = N_SERIES_0 - int((catalogo_ia["type"] == "TV Show").sum())
print(f"títulos borrados : {N_TITULOS_0 - len(catalogo_ia):,} ({(N_TITULOS_0 - len(catalogo_ia)) / N_TITULOS_0:.1%})")
print(f"series borradas  : {series_perdidas:,} de {N_SERIES_0:,} ({series_perdidas / N_SERIES_0:.1%})")
print(f"el catálogo pasa de {N_SERIES_0 / N_TITULOS_0:.1%} de series a "
      f"{(catalogo_ia['type'] == 'TV Show').mean():.1%}")

**De 2 410 series quedan 131.** El fragmento borró el 38,3 % del catálogo y el 94,6 % de las
series, sin un error, sin un warning, con un mensaje final que decía «impecable». Cualquier análisis
posterior —¿cuánto contenido añadimos por año?, ¿qué peso tienen las series?— saldría
espectacularmente sesgado, porque `dropna()` no borró al azar: borró justo donde el nulo significaba
«no aplica». Un `dropna()` global no es limpiar; es amputar sin mirar. La versión honesta se escribe
con `subset` y después de leer la sección 5.

Fíjate además en qué **no** detectó el fragmento: el duplicado de *Consequences* sobrevivió a
`drop_duplicates()`, porque su espacio invisible lo hace «distinto». El código IA borró 2 983 filas
que no debía y conservó la única que sí sobraba. Por eso la bitácora y la cifra de control no son
burocracia: son lo único que te permite mirar un «impecable» y contestar *«demuéstralo»*.

In [ ]:
# ✅ Comprobación 2 · el daño del fragmento queda cuantificado con estas cifras
assert len(catalogo_ia) == 4_804, \
    f"El fragmento debería dejar 4 804 títulos y deja {len(catalogo_ia):,}: ejecútalo tal cual, sin arreglarlo todavía"
assert int((catalogo_ia["type"] == "TV Show").sum()) == 131, \
    "Deberían sobrevivir exactamente 131 series al dropna(): revisa que partiste del archivo crudo"

print("Comprobación 2 superada ✓  dropna() borró 2 983 títulos y el 94,6 % de las series, y tú lo mediste")

## Entregable

Sube `lab_04_apellido.ipynb` con:

- El inventario de calidad completo, con el porcentaje exacto de cada problema.
- La bitácora de limpieza con las ocho filas, más las que añadas en el ejercicio guiado. Cada fila
  con su decisión **y** su justificación en una línea.
- La cifra de control: la tabla de invariantes con el único cambio explicado.
- El daño del fragmento generado por IA cuantificado: títulos perdidos, % de series perdidas y
  fechas reales borradas por el `coerce`.
- Una fila nueva en la bitácora de prompts: el prompt con el que le pediste al asistente que
  revisara tu propia limpieza, y qué encontró que tú no habías visto.

## Para tu equipo

- La bitácora de limpieza del dataset del caso es entregable de esta semana. La columna de
  justificación es la que se califica: cualquiera elimina duplicados, casi nadie escribe por qué.
- Antes de limpiar, guarden las cifras de partida —filas, claves distintas, conteos por categoría—.
  Sin ellas no hay cifra de control posible y la limpieza no se puede auditar.
- Pregunten por cada columna con nulos **dónde viven** esos nulos antes de decidir. Si el hueco tiene
  estructura, como el director de las series, borrarlo o imputarlo fabrica un sesgo.

---

📌 **Lo que viene.** Con hoy cierras el circuito de entrenamiento: una pizzería te enseñó que la
métrica decide el ranking, un retail que la unidad decide el conteo, un catálogo musical que la forma
decide el estadístico, y uno de streaming que ningún número sobrevive a datos sin auditar. Desde la
**semana 5** entra **Comercial Andina**, el negocio integrado del curso: un distribuidor ecuatoriano
con seis sucursales, 80 515 líneas de venta y todos los problemas de estas cuatro semanas juntos en
los mismos archivos. Su tabla de ventas ya pasó por una limpieza exactamente como la de hoy —por eso
existe `ventas_limpias.csv`—, y sobre ella vas a agrupar, combinar y construir los indicadores del
resto del curso. A partir de ahora, cuando leas «limpias» en un nombre de archivo, ya sabes todo lo
que esa palabra promete y todo lo que hay que auditar antes de creérsela.